# **Practical — Recurrent Neural Networks**

**RNN · BPTT · LSTM · GRU · BiLSTM in PyTorch (CPU only, no datasets)**

### Aim
1. **Experiment 1:** Run a scalar RNN by hand, build the cell from `nn.Linear` layers and count its parameters.
2. **Experiment 2:** Derive BPTT and check it against autograd and a finite difference; measure vanishing and exploding gradients; apply clipping and truncated BPTT.
3. **Experiment 3:** Trace an LSTM step by hand, measure the effect of the forget-gate bias on gradient reach, and train a 60-step recall task.
4. **Experiment 4:** Rebuild `nn.GRUCell` from its equations and count the parameters of every cell, with and without bidirectionality.
5. **Experiment 5:** Show that a BiLSTM is two LSTMs, then test it on a task that needs the next token and on a simulated stream.

### Datasets
None. Every sequence is generated inside the notebook.

### Notation
$\odot$ is the element-wise product, $[a;b]$ is concatenation, and superscripts $f$ and $b$ mark the forward and backward directions.

## **Step 0 — Setup**

Only `torch` is needed. Colab already has it; elsewhere use `pip install torch`. Put each code box in its own cell and run them **in order**: later cells reuse `reach`, `make_rnn`, `cell_params` and other names defined earlier.

In [1]:
# -------------------- IMPORT LIBRARIES --------------------

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# Fix the random number generator so results repeat from run to run
torch.manual_seed(0)

print("PyTorch", torch.__version__)

PyTorch 2.11.0+cu128


# **Experiment 1 — RNN: Forward Pass and Parameter Count**

**Aim:** Run a scalar RNN for two steps by hand, then build the cell from `nn.Linear` layers, count its parameters and run the same weights over 5 and 5,000 steps.

A recurrent cell reads one input at a time and carries a hidden state forward. The **same weights** are used at every step, so the parameter count does not depend on the sequence length.

$$h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h), \qquad o_t = W_{hy}h_t + b_y$$

## **1.1 — Forward Pass by Hand**

A scalar RNN with a tanh hidden unit and a sigmoid output, run for two steps.

In [2]:
# Scalar RNN: tanh hidden unit, sigmoid output
wx, wh, b, wy, c = 0.5, 0.4, 0.1, 1.0, 0.0

# Hidden state starts at zero
h = 0.0

for t, x in enumerate([1.0, 2.0], start=1):
    a = wx * x + wh * h + b                # pre-activation
    h = math.tanh(a)                       # new hidden state
    y = 1 / (1 + math.exp(-(wy * h + c)))  # sigmoid output

    print(f"t={t}  a={a:.4f}  h={h:.4f}  y={y:.4f}")

t=1  a=0.6000  h=0.5370  y=0.6311
t=2  a=1.3148  h=0.8655  y=0.7038



- The second hidden state depends on the first through `wh`. Setting `wh = 0` makes the two steps independent: the model loses its memory.

## **1.2 — The Cell from Scratch**

The cell is built from `nn.Linear` layers. `Wxh` has no bias and `Whh` carries the single hidden bias $b_h$, so the cell matches the equations exactly.

In [4]:
class RNNCell(nn.Module):
    def __init__(self, dx, dh, dy):
        super().__init__()
        self.Wxh = nn.Linear(dx, dh, bias=False)   # input  -> hidden
        self.Whh = nn.Linear(dh, dh, bias=True)    # hidden -> hidden, plus b_h
        self.Why = nn.Linear(dh, dy)               # hidden -> output, plus b_y

    def forward(self, x, h):
        h = torch.tanh(self.Wxh(x) + self.Whh(h))
        return h, self.Why(h)


cell = RNNCell(dx=3, dh=4, dy=2)

# Count every weight and bias in the cell
print("parameters:", sum(p.numel() for p in cell.parameters()))

with torch.no_grad():
    # The SAME weights are used for any sequence length
    for T in (5, 5000):
        h = torch.zeros(1, 4)

        for x in torch.randn(T, 1, 3):
            h, o = cell(x, h)

        print("T =", T, "-> h shape", tuple(h.shape))

parameters: 42
T = 5 -> h shape (1, 4)
T = 5000 -> h shape (1, 4)


**Observation:**
- Expected: `parameters: 42`, and the hidden state has shape `(1, 4)` for both `T = 5` and `T = 5000`.
- The count follows from the weight shapes, for $d_x=3$, $d_h=4$, $d_y=2$: $12 + 16 + 4 + 8 + 2 = 42$, whatever $T$ is.

$$N = d_hd_x + d_h^2 + d_h + d_yd_h + d_y$$

- **Note:** `nn.RNN` keeps *two* hidden bias vectors (`bias_ih` and `bias_hh`), so it reports $d_h$ more parameters per direction than this formula.

# **Experiment 2 — BPTT: Gradients Through Time**

**Aim:** Write backpropagation through time by hand and verify it, measure how far gradients travel through a plain RNN, then apply clipping and truncation.

BPTT is ordinary backpropagation on the unrolled network. Weights are shared, so their gradients are **sums with one term per step**. For a softmax output with cross-entropy loss at every step, the sweep from $t=T$ down to $t=1$ is:

$$\delta_t = \left(W_{hy}^T(\hat y_t - y_t) + W_{hh}^T\delta_{t+1}\right)\odot(1-h_t^2), \qquad \frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T}\delta_t h_{t-1}^T$$

The derivative of tanh is $1-h_t^2$, read straight off the stored state, and it never exceeds 1.

## **2.1 — Scalar BPTT, Three Ways**

A two-step scalar RNN ($w_x=1$, $w_h=0.5$, $h_0=0$, inputs 1 and 1) with a squared-error loss on the last step. The gradient of $w_h$ is computed **by hand**, with **autograd**, and with a **central finite difference**.

In [5]:
w_x, w_h, y = 1.0, 0.5, 0.8
inputs = [1.0, 1.0]

# ---- forward pass, storing every hidden state ----
hs = [0.0]
for x in inputs:
    hs.append(math.tanh(w_x * x + w_h * hs[-1]))

print(f"h_1 = {hs[1]:.4f}  h_2 = {hs[2]:.4f}  loss = {0.5 * (hs[2] - y) ** 2:.6f}")

# ---- backward pass by hand: t = 2, then t = 1 ----
delta2 = (hs[2] - y) * (1 - hs[2] ** 2)
delta1 = (w_h * delta2) * (1 - hs[1] ** 2)

# w_h is used at both steps, so its gradient is a SUM of two terms
grad = delta1 * hs[0] + delta2 * hs[1]

print(f"delta_2 = {delta2:.6f}   delta_1 = {delta1:.6f}")
print(f"dL/dw_h by hand      = {grad:.6f}")

# ---- the same gradient from autograd ----
w = torch.tensor(0.5, requires_grad=True)
h = torch.tensor(0.0)

for x in inputs:
    h = torch.tanh(w_x * x + w * h)

(0.5 * (h - y) ** 2).backward()
print(f"dL/dw_h by autograd  = {w.grad.item():.6f}")


# ---- and from a central finite difference ----
def loss_for(wv):
    h = 0.0
    for x in inputs:
        h = math.tanh(w_x * x + wv * h)
    return 0.5 * (h - y) ** 2


eps = 1e-5
print(f"central difference   = {(loss_for(0.5 + eps) - loss_for(0.5 - eps)) / (2 * eps):.6f}")

h_1 = 0.7616  h_2 = 0.8811  loss = 0.003291
delta_2 = 0.018141   delta_1 = 0.003809
dL/dw_h by hand      = 0.013816
dL/dw_h by autograd  = 0.013816
central difference   = 0.013816


## **2.2 — The Full Matrix Pass**

A 2-input, 2-hidden, 2-class RNN over four steps with a softmax output and a loss at every step. Both sweeps are written from the equations above and every gradient is compared with autograd.

In [6]:
W_xh = torch.tensor([[0.5, -0.3], [0.8, 0.2]])
W_hh = torch.tensor([[0.1, 0.4], [-0.2, 0.3]])
W_hy = torch.tensor([[1.0, -1.0], [0.5, 0.5]])

X = [torch.tensor(v) for v in ([1., 0.], [0., 1.], [1., 1.], [0.5, -0.5])]
targets = [0, 1, 0, 1]


def bptt(X, targets):
    H, P, loss = [torch.zeros(2)], [], 0.0

    # ---- forward sweep: store every hidden state and probability ----
    for t in range(len(X)):
        h = torch.tanh(W_xh @ X[t] + W_hh @ H[-1])
        p = F.softmax(W_hy @ h, dim=0)
        H.append(h)
        P.append(p)
        loss += -math.log(p[targets[t]].item())

    delta_next = torch.zeros(2)
    g_hh, g_xh, g_hy = (torch.zeros_like(W) for W in (W_hh, W_xh, W_hy))

    # ---- backward sweep: t = T-1 down to 0 ----
    for t in reversed(range(len(X))):
        d_o = P[t] - F.one_hot(torch.tensor(targets[t]), 2).float()   # y_hat - y

        g_hy += torch.outer(d_o, H[t + 1])

        # note the TRANSPOSES: W_hy.T and W_hh.T
        delta = (W_hy.T @ d_o + W_hh.T @ delta_next) * (1 - H[t + 1] ** 2)

        g_hh += torch.outer(delta, H[t])      # summed over time
        g_xh += torch.outer(delta, X[t])

        delta_next = delta

    return loss, g_hh, g_xh, g_hy


loss, g_hh, g_xh, g_hy = bptt(X, targets)

# ---- the same model, differentiated by autograd ----
P_ = [W.clone().requires_grad_(True) for W in (W_xh, W_hh, W_hy)]
h, total = torch.zeros(2), 0.0

for t in range(len(X)):
    h = torch.tanh(P_[0] @ X[t] + P_[1] @ h)
    total = total + F.cross_entropy((P_[2] @ h)[None], torch.tensor([targets[t]]))

total.backward()

print(f"hand loss {loss:.4f}    autograd loss {total.item():.4f}")

for name, mine, p in zip(("W_xh", "W_hh", "W_hy"), (g_xh, g_hh, g_hy), P_):
    print(f"{name} matches autograd:", torch.allclose(mine, p.grad, atol=1e-5))

hand loss 3.5200    autograd loss 3.5200
W_xh matches autograd: True
W_hh matches autograd: True
W_hy matches autograd: True


**Observation:**
- Expected: both losses are **3.5200** and all three gradients match autograd (`True`).
- **Read the shapes:** $\delta_t$ and $h_{t-1}$ are both 2-vectors, so their outer product is 2 × 2, the shape of $W_{hh}$. A shape check like this catches most transcription errors before any numbers are compared.
- Note `W_hh.T @ delta_next`: this $W_{hh}$ is **not symmetric**, so leaving the transpose out gives a wrong gradient that is hard to spot by eye (Exercise 2).

## **2.3 — Vanishing and Exploding Gradients, and Clipping**

`reach` sends a loss at the **last** step back to the **first** input and measures how large the gradient is when it arrives. The default recurrent matrix is compared with one scaled by 3, then a gradient vector is clipped.

In [7]:
def reach(module, T, seed=0):
    # Gradient of the LAST output with respect to the FIRST input
    torch.manual_seed(seed)
    x = torch.randn(1, T, 1, requires_grad=True)

    out, _ = module(x)
    out[0, -1].sum().backward()

    # How much of the signal from step T arrived at step 0
    return x.grad[0, 0].norm().item()


def make_rnn(scale=1.0):
    torch.manual_seed(0)
    m = nn.RNN(1, 16, batch_first=True)

    with torch.no_grad():
        m.weight_hh_l0.mul_(scale)       # inflate the recurrent matrix

    return m


print(f"{'T':>4} {'scale 1.0':>12} {'scale 3.0':>12}")

for T in (5, 10, 20, 40):
    print(f"{T:>4} {reach(make_rnn(), T):>12.2e} {reach(make_rnn(3.0), T):>12.2e}")


# ---- clipping: rescale the length, keep the direction ----
w = nn.Parameter(torch.zeros(2))
w.grad = torch.tensor([9.0, 12.0])                    # norm 15

returned = nn.utils.clip_grad_norm_([w], max_norm=5.0)

print(f"\nnorm before {float(returned):.1f}, after {w.grad.norm().item():.1f}",
      "| direction kept:", torch.allclose(w.grad / w.grad.norm(), torch.tensor([0.6, 0.8])))

   T    scale 1.0    scale 3.0
   5     1.47e-02     7.66e-03
  10     1.20e-03     7.64e-01
  20     7.22e-07     1.66e+00
  40     8.64e-12     2.40e+00

norm before 15.0, after 5.0 | direction kept: True


## **2.4 — Truncated BPTT**

The forward pass runs over 20 steps, but the hidden state is **detached every 5 steps**, so the backward pass is cut into four chunks (1–5, 6–10, 11–15, 16–20). The result is compared with the full gradient.

In [8]:
T, k, d_h = 20, 5, 4

torch.manual_seed(0)
Wx0 = torch.randn(d_h, 1) * 0.5
Wh0 = torch.randn(d_h, d_h) * 0.3
seq = torch.randn(T, 1)


def run(truncate):
    Wx = Wx0.clone().requires_grad_(True)
    Wh = Wh0.clone().requires_grad_(True)

    h, total = torch.zeros(d_h), 0.0

    for t in range(T):
        if truncate and t % k == 0:
            h = h.detach()                 # keep the value, cut the gradient path

        h = torch.tanh(Wx @ seq[t] + Wh @ h)
        total = total + h.pow(2).sum()

    total.backward()
    return Wh.grad.clone()


g_full, g_trunc = run(False), run(True)

print("full      ||dL/dW_hh|| =", round(g_full.norm().item(), 4))
print("truncated ||dL/dW_hh|| =", round(g_trunc.norm().item(), 4))

cos = F.cosine_similarity(g_full.flatten(), g_trunc.flatten(), dim=0)
print("cosine similarity      =", round(cos.item(), 4))

full      ||dL/dW_hh|| = 5.7002
truncated ||dL/dW_hh|| = 5.8338
cosine similarity      = 0.9945


# **Experiment 3 — LSTM: Gates and Long-Range Memory**

**Aim:** Trace one LSTM step by hand, show that the forget-gate bias controls how far gradients travel, and train a task that cannot be solved without long-range memory.

The LSTM adds a cell state $c_t$ that is updated **additively**, with gates deciding what to forget, what to write and what to show. The input gate $i_t$ and output gate $o_t$ have the same form as $f_t$ with their own weights. The candidate $g_t$ uses tanh because new content may need to be negative.

$$f_t=\sigma(W_f[h_{t-1};x_t]+b_f),\quad g_t=\tanh(W_c[h_{t-1};x_t]+b_c),\quad c_t=f_t\odot c_{t-1}+i_t\odot g_t,\quad h_t=o_t\odot\tanh(c_t)$$

## **3.1 — The Scalar Forward Pass**

A scalar LSTM with $x=1$, $h_{t-1}=0.2$ and $c_{t-1}=0.5$. Every gate reads the same input $z = 1.2$; only the weights differ.

In [9]:
def sigmoid(z):
    return 1 / (1 + math.exp(-z))


z, c_prev = 1.2, 0.5                        # z = h_{t-1} + x_t = 0.2 + 1.0

f = sigmoid(0.5 * z + 0.1)                  # forget gate
i = sigmoid(0.4 * z + 0.0)                  # input gate
c_tilde = math.tanh(0.3 * z + 0.1)          # candidate (tanh: may be negative)
o = sigmoid(0.6 * z - 0.1)                  # output gate

c = f * c_prev + i * c_tilde                # the memory equation: ADD, not compose
h = o * math.tanh(c)

print(f"f={f:.4f}  i={i:.4f}  c~={c_tilde:.4f}  o={o:.4f}")
print(f"c_t = {f * c_prev:.4f} + {i * c_tilde:.4f} = {c:.4f}")
print(f"h_t = {o:.4f} * tanh({c:.4f}) = {h:.4f}")

f=0.6682  i=0.6177  c~=0.4301  o=0.6502
c_t = 0.3341 + 0.2657 = 0.5998
h_t = 0.6502 * tanh(0.5998) = 0.3491


## **3.2 — Forget-Gate Bias and Gradient Reach**

The measurement of Experiment 2.3 is repeated for a plain RNN and for LSTMs whose forget-gate bias starts at 0 or 2. PyTorch orders the LSTM gates **i, f, g, o**, so the forget slice of each bias is the **second block** of $d_h$ entries.

In [10]:
def lstm_fb(bias, d_h=16):
    torch.manual_seed(0)
    m = nn.LSTM(1, d_h, batch_first=True)

    with torch.no_grad():
        # the f slice (PyTorch order: i, f, g, o)
        m.bias_ih_l0[d_h:2 * d_h].fill_(bias)
        m.bias_hh_l0[d_h:2 * d_h].fill_(0.0)

    return m


print(f"{'T':>4} {'RNN':>10} {'LSTM b=0':>10} {'LSTM b=2':>10}")

for T in (10, 20, 40, 80):
    print(f"{T:>4} {reach(make_rnn(), T):>10.2e} {reach(lstm_fb(0.0), T):>10.2e}"
          f" {reach(lstm_fb(2.0), T):>10.2e}")

   T        RNN   LSTM b=0   LSTM b=2
  10   1.20e-03   2.00e-04   5.62e-03
  20   7.22e-07   1.32e-06   1.83e-02
  40   8.64e-12   2.50e-10   3.08e-03
  80   7.82e-23   1.20e-18   2.42e-03


## **3.3 — A Task That Needs Memory**

One bit appears at step 0, the next 59 steps are noise, and the model must report the bit at the end. A plain RNN, a default LSTM and an LSTM with forget bias 2 are trained on it with the same data and settings.

In [11]:
def make_batch(n, T, gen):
    # A bit at step 0, noise afterwards; report the bit at the end
    x = torch.zeros(n, T, 2)
    bits = torch.randint(0, 2, (n,), generator=gen).float()

    x[:, 0, 0] = bits                                        # channel 0: the signal
    x[:, 1:, 1] = torch.randn(n, T - 1, generator=gen) * 0.5   # channel 1: noise

    return x, bits


class Recall(nn.Module):
    def __init__(self, kind, forget_bias=0.0, d_h=24):
        super().__init__()
        self.rnn = (nn.LSTM if kind == "lstm" else nn.RNN)(2, d_h, batch_first=True)

        if kind == "lstm":
            with torch.no_grad():
                self.rnn.bias_ih_l0[d_h:2 * d_h].fill_(forget_bias)
                self.rnn.bias_hh_l0[d_h:2 * d_h].fill_(0.0)

        self.fc = nn.Linear(d_h, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1]).squeeze(-1)     # read the LAST step only


T = 60

for label, kind, fb in [("RNN", "rnn", 0.0),
                        ("LSTM, default", "lstm", 0.0),
                        ("LSTM, forget bias 2", "lstm", 2.0)]:

    torch.manual_seed(0)
    gen = torch.Generator().manual_seed(1)

    model = Recall(kind, fb)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)

    for step in range(400):
        x, y = make_batch(64, T, gen)

        opt.zero_grad()
        F.binary_cross_entropy_with_logits(model(x), y).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

    # Fresh test data from a different generator seed
    x, y = make_batch(512, T, torch.Generator().manual_seed(99))

    with torch.no_grad():
        acc = ((model(x) > 0).float() == y).float().mean().item()

    print(f"{label:>20}: test accuracy {100 * acc:5.1f}%")

                 RNN: test accuracy  68.9%
       LSTM, default: test accuracy  51.0%
 LSTM, forget bias 2: test accuracy 100.0%


# **Experiment 4 — GRU: Fewer Gates, Same Idea**

**Aim:** Rebuild `nn.GRUCell` from its equations, find where the textbook GRU and PyTorch's differ, and count the parameters of every recurrent cell.

The GRU drops the separate cell state and the output gate. Its hidden state is the memory: an update gate $z_t$ blends the old state with a candidate $n_t$, and a reset gate $r_t$ controls how much of the old state the candidate may see. Textbook form ($r_t$ has the same form as $z_t$):

$$z_t=\sigma(W_z[h_{t-1};x_t]+b_z),\quad n_t=\tanh(W_h[r_t\odot h_{t-1};x_t]+b_h),\quad h_t=(1-z_t)\odot h_{t-1}+z_t\odot n_t$$

## **4.1 — The Textbook GRU against `nn.GRUCell`**

The weight blocks are pulled out of a library `GRUCell` and its output is rebuilt from the equations. If they describe the library, the difference should be float32 noise (about 1e-8).

In [12]:
torch.manual_seed(0)
d_x, d_h = 3, 4

gru = nn.GRUCell(d_x, d_h)
x, h_prev = torch.randn(1, d_x), torch.randn(1, d_h)

ref = gru(x, h_prev)                              # the library's answer

# PyTorch stacks the three blocks in the order r, z, n
W_ir, W_iz, W_in = gru.weight_ih.chunk(3, 0)
W_hr, W_hz, W_hn = gru.weight_hh.chunk(3, 0)
b_ir, b_iz, b_in = gru.bias_ih.chunk(3, 0)
b_hr, b_hz, b_hn = gru.bias_hh.chunk(3, 0)

r = torch.sigmoid(x @ W_ir.T + b_ir + h_prev @ W_hr.T + b_hr)
z = torch.sigmoid(x @ W_iz.T + b_iz + h_prev @ W_hz.T + b_hz)

# TEXTBOOK: reset acts BEFORE the matrix, z weights the candidate
n_book = torch.tanh(x @ W_in.T + b_in + (r * h_prev) @ W_hn.T + b_hn)
h_book = (1 - z) * h_prev + z * n_book

# PYTORCH: reset acts AFTER the matrix, z weights the OLD state
n_torch = torch.tanh(x @ W_in.T + b_in + r * (h_prev @ W_hn.T + b_hn))
h_torch = (1 - z) * n_torch + z * h_prev


def err(a):
    return (ref - a).abs().max().item()


print(f"textbook equations     : max error {err(h_book):.4f}")
print(f"PyTorch's two variants : max error {err(h_torch):.1e}")

textbook equations     : max error 0.3462
PyTorch's two variants : max error 3.0e-08


## **4.2 — Parameter Counts for Every Cell**

Every cell is a stack of $k$ affine transformations of $[h_{t-1};x_t]$, with $k=1$ for the RNN, 3 for the GRU and 4 for the LSTM:

$$P = k\,d_h(d_x + d_h + 1)$$

The formula is compared with PyTorch for $d_x=10$, $d_h=20$, with and without bidirectionality.

In [13]:
def cell_params(sets, d_x, d_h):
    # one weight set per affine transformation
    return sets * d_h * (d_x + d_h + 1)


d_x, d_h = 10, 20

print(f"{'model':>7} {'formula':>8} {'PyTorch':>8} {'excess':>7}")

for name, cls, sets in (("RNN", nn.RNN, 1), ("GRU", nn.GRU, 3), ("LSTM", nn.LSTM, 4)):
    for bi in (False, True):
        m = cls(d_x, d_h, bidirectional=bi)

        # bidirectional = two copies of the cell
        formula = cell_params(sets, d_x, d_h) * (2 if bi else 1)
        total = sum(p.numel() for p in m.parameters())

        label = ("Bi" if bi else "") + name
        print(f"{label:>7} {formula:>8} {total:>8} {total - formula:>7}")

  model  formula  PyTorch  excess
    RNN      620      640      20
  BiRNN     1240     1280      40
    GRU     1860     1920      60
  BiGRU     3720     3840     120
   LSTM     2480     2560      80
 BiLSTM     4960     5120     160


# **Experiment 5 — BiLSTM: Past and Future Context**

**Aim:** Show that `nn.LSTM(bidirectional=True)` is exactly two LSTMs, then test it on a task that needs the next token, and finally on a simulated live stream.

A BiLSTM is two ordinary LSTMs: one reads left to right, the other right to left, and their hidden states are joined at every position. The subscript $t$ always means position $t$ of the original sequence.

$$h_t^f=\text{LSTM}(x_t,h_{t-1}^f,c_{t-1}^f),\quad h_t^b=\text{LSTM}(x_t,h_{t+1}^b,c_{t+1}^b),\quad h_t=[h_t^f;h_t^b],\quad P_{BiLSTM}=8d_h(d_x+d_h+1)$$

## **5.1 — Two LSTMs in One Module**

The forward and `_reverse` weights of a bidirectional LSTM are copied into two ordinary LSTMs. The second runs on the **flipped** sequence and its output is flipped back, then the two are concatenated and compared. Then the cell checks which numbers live where.

In [14]:
d_x, d_h, T = 3, 4, 6

torch.manual_seed(0)
bi = nn.LSTM(d_x, d_h, batch_first=True, bidirectional=True)
x = torch.randn(1, T, d_x)

out, (h_n, c_n) = bi(x)
print("out", tuple(out.shape), "  h_n", tuple(h_n.shape))

# Two ordinary one-direction LSTMs
fwd, bwd = (nn.LSTM(d_x, d_h, batch_first=True) for _ in range(2))

with torch.no_grad():
    # Copy the forward weights into fwd and the "_reverse" weights into bwd
    for n in ("weight_ih_l0", "weight_hh_l0", "bias_ih_l0", "bias_hh_l0"):
        getattr(fwd, n).copy_(getattr(bi, n))
        getattr(bwd, n).copy_(getattr(bi, n + "_reverse"))

    out_f, _ = fwd(x)
    out_b, _ = bwd(x.flip(1))                 # read right to left ...
    out_b = out_b.flip(1)                     # ... then restore position order

    manual = torch.cat([out_f, out_b], dim=2)
    print("max |BiLSTM - two LSTMs| =", (out - manual).abs().max().item())

    # Which numbers live where?
    ok_f = torch.allclose(h_n[0], out[:, -1, :d_h])
    ok_b = torch.allclose(h_n[1], out[:, 0, d_h:])
    print("h_n[0] is the forward state at the last position  :", ok_f)
    print("h_n[1] is the backward state at the first position:", ok_b)

    # The correct whole-sequence vector, and how wrong out[:, -1] is
    whole = torch.cat([h_n[0], h_n[1]], dim=1)
    rel = ((whole - out[:, -1]).norm() / whole.norm()).item()
    print(f"out[:, -1] is {100 * rel:.0f}% away from the true sequence vector")

out (1, 6, 8)   h_n (2, 1, 4)
max |BiLSTM - two LSTMs| = 2.9802322387695312e-08
h_n[0] is the forward state at the last position  : True
h_n[1] is the backward state at the first position: True
out[:, -1] is 58% away from the true sequence vector


## **5.2 — A Task That Needs the Next Token**

Each position gets a random bit and the label at position $t$ is the bit at position $t+1$. Nothing in the past predicts it. A unidirectional LSTM and a BiLSTM are trained, then the trained BiLSTM is deployed **as if on a stream**: at position $t$, every later input is replaced by zeros.

In [15]:
def make_tag_batch(n, T, gen):
    # Label at position t is the token at position t + 1
    tok = torch.randint(0, 2, (n, T), generator=gen).float()

    y = torch.zeros(n, T)
    y[:, :-1] = tok[:, 1:]            # shift left by one

    return tok.unsqueeze(-1), y


class Tagger(nn.Module):
    def __init__(self, bidirectional, d_h=16):
        super().__init__()
        self.rnn = nn.LSTM(1, d_h, batch_first=True, bidirectional=bidirectional)
        self.fc = nn.Linear(d_h * (2 if bidirectional else 1), 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out).squeeze(-1)           # one logit per position


T = 12


def train_tagger(model, steps=300):
    gen = torch.Generator().manual_seed(1)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)

    for _ in range(steps):
        x, y = make_tag_batch(64, T, gen)

        opt.zero_grad()
        # the last position has no next token, so it is left out of the loss
        F.binary_cross_entropy_with_logits(model(x)[:, :-1], y[:, :-1]).backward()
        opt.step()

    return model


def score_tagger(model):
    x, y = make_tag_batch(512, T, torch.Generator().manual_seed(99))

    with torch.no_grad():
        return ((model(x)[:, :-1] > 0).float() == y[:, :-1]).float().mean().item()


for label, bi in (("unidirectional LSTM", False), ("BiLSTM", True)):
    torch.manual_seed(0)
    model = train_tagger(Tagger(bi))
    print(f"{label:>20}: test accuracy {100 * score_tagger(model):5.1f}%")


# ---- Deploy the trained BiLSTM on a stream: at position t the future is zero-padded ----
x, y = make_tag_batch(512, T, torch.Generator().manual_seed(99))
hits = total = 0

with torch.no_grad():
    for t in range(T - 1):
        seen = x.clone()
        seen[:, t + 1:, :] = 0.0                   # hide everything after position t

        hits += ((model(seen)[:, t] > 0).float() == y[:, t]).sum().item()
        total += y.shape[0]

print(f"{'BiLSTM on a stream':>20}: test accuracy {100 * hits / total:5.1f}%")

 unidirectional LSTM: test accuracy  50.0%
              BiLSTM: test accuracy 100.0%
  BiLSTM on a stream: test accuracy  49.0%


# **Lab Exercises**

Complete these in the same notebook. **Where an exercise asks for a prediction, write it down before running the cell.** They reuse `reach`, `make_rnn`, `lstm_fb`, `make_batch`, `Recall`, `make_tag_batch`, `Tagger`, `train_tagger` and the weights from the experiments above, so run those cells first. Fill in the **Observation** cell of each exercise from your own output.

## **Exercise 1 — Scalar BPTT with $h_0 = 0.5$ (Experiment 2.1)**

**Prediction first:** do the three gradients still agree, and which term of the sum was previously zero?

**Prediction:** Yes, all three still agree. The $t=1$ term $\delta_1 h_0$ was zero only because $h_0=0$; with $h_0=0.5$ it becomes non-zero, so the gradient is larger and now **tests $\delta_1$** as well.

In [16]:
w_x, w_h, y = 1.0, 0.5, 0.8
inputs = [1.0, 1.0]
h0 = 0.5                                   # was 0.0

# ---- forward pass ----
hs = [h0]
for x in inputs:
    hs.append(math.tanh(w_x * x + w_h * hs[-1]))

# ---- backward pass by hand ----
delta2 = (hs[2] - y) * (1 - hs[2] ** 2)
delta1 = (w_h * delta2) * (1 - hs[1] ** 2)

term1 = delta1 * hs[0]                     # the t = 1 term, now non-zero
term2 = delta2 * hs[1]                     # the t = 2 term
print(f"t=1 term = {term1:.6f}   t=2 term = {term2:.6f}")
print(f"dL/dw_h by hand      = {term1 + term2:.6f}")

# ---- autograd ----
w = torch.tensor(0.5, requires_grad=True)
h = torch.tensor(h0)
for x in inputs:
    h = torch.tanh(w_x * x + w * h)
(0.5 * (h - y) ** 2).backward()
print(f"dL/dw_h by autograd  = {w.grad.item():.6f}")


# ---- central finite difference ----
def loss_for(wv):
    h = h0
    for x in inputs:
        h = math.tanh(w_x * x + wv * h)
    return 0.5 * (h - y) ** 2


eps = 1e-5
print(f"central difference   = {(loss_for(0.5 + eps) - loss_for(0.5 - eps)) / (2 * eps):.6f}")

t=1 term = 0.001313   t=2 term = 0.015891
dL/dw_h by hand      = 0.017204
dL/dw_h by autograd  = 0.017204
central difference   = 0.017204


**Observation:** Prediction held. All three gradients agree at 0.017204. The t=1 term, previously zero because h₀ = 0, is now 0.001313 and the t=2 term is 0.015891, so the answer now depends on δ₁ as well and the check is a real test of it.

## **Exercise 2 — Remove the transpose in the backward sweep (Experiment 2.2)**

**Prediction first:** does the match with autograd survive if `W_hh` is used instead of `W_hh.T`? What happens when `W_hh` is made symmetric?

**Prediction:** The match **breaks** for `W_hh` and `W_xh`, because both are built from $\delta_t$ and $\delta_t$ now carries the wrong matrix. `W_hy` is computed before $\delta$ and should still match. With a **symmetric** $W_{hh}$, $W_{hh}^T = W_{hh}$, so everything matches again.

In [17]:
def bptt_variant(W_hh, use_transpose):
    # Same sweeps as bptt() in 2.2, but W_hh is a parameter and the transpose is optional
    H, P, loss = [torch.zeros(2)], [], 0.0

    for t in range(len(X)):
        h = torch.tanh(W_xh @ X[t] + W_hh @ H[-1])
        p = F.softmax(W_hy @ h, dim=0)
        H.append(h)
        P.append(p)
        loss += -math.log(p[targets[t]].item())

    delta_next = torch.zeros(2)
    g_hh, g_xh, g_hy = (torch.zeros_like(W) for W in (W_hh, W_xh, W_hy))

    for t in reversed(range(len(X))):
        d_o = P[t] - F.one_hot(torch.tensor(targets[t]), 2).float()
        g_hy += torch.outer(d_o, H[t + 1])

        M = W_hh.T if use_transpose else W_hh        # <-- the line under test
        delta = (W_hy.T @ d_o + M @ delta_next) * (1 - H[t + 1] ** 2)

        g_hh += torch.outer(delta, H[t])
        g_xh += torch.outer(delta, X[t])
        delta_next = delta

    return g_xh, g_hh, g_hy


def autograd_grads(W_hh):
    P_ = [W.clone().requires_grad_(True) for W in (W_xh, W_hh, W_hy)]
    h, total = torch.zeros(2), 0.0

    for t in range(len(X)):
        h = torch.tanh(P_[0] @ X[t] + P_[1] @ h)
        total = total + F.cross_entropy((P_[2] @ h)[None], torch.tensor([targets[t]]))

    total.backward()
    return [p.grad for p in P_]


W_sym = torch.tensor([[0.1, 0.4], [0.4, 0.3]])        # symmetric: W == W.T

print(f"{'W_hh':>10} {'transpose':>10} {'W_xh':>6} {'W_hh':>6} {'W_hy':>6}")

for name, W in (("asymmetric", W_hh), ("symmetric", W_sym)):
    ref = autograd_grads(W)

    for use_t in (True, False):
        mine = bptt_variant(W, use_t)
        ok = [torch.allclose(m, r, atol=1e-5) for m, r in zip(mine, ref)]
        print(f"{name:>10} {str(use_t):>10} {str(ok[0]):>6} {str(ok[1]):>6} {str(ok[2]):>6}")

      W_hh  transpose   W_xh   W_hh   W_hy
asymmetric       True   True   True   True
asymmetric      False  False  False   True
 symmetric       True   True   True   True
 symmetric      False   True   True   True


**Observation:** Prediction held. With the asymmetric W_hh and no transpose, W_xh and W_hh no longer match autograd, while W_hy still matches because it is computed before δ. With the symmetric W_hh, W_hhᵀ = W_hh, so all three match with or without the transpose.

## **Exercise 3 — Forget biases 1, 2, 4 and 6 at T = 80 (Experiment 3.2)**

**Prediction first:** is a larger forget bias always better?

**Prediction:** No. A larger bias keeps the gate closer to 1, so the gradient path stays open, but a gate stuck near 1 **never forgets**, so the cell state saturates and the model cannot clear old content. I expect reach to rise from bias 1 to about 2 and then flatten or fall for 4 and 6.

In [18]:
print(f"{'forget bias':>12} {'sigmoid(b)':>11} {'reach at T=80':>14}")

for b in (0.0, 1.0, 2.0, 4.0, 6.0):
    print(f"{b:>12.1f} {torch.sigmoid(torch.tensor(b)).item():>11.4f} "
          f"{reach(lstm_fb(b), 80):>14.2e}")

 forget bias  sigmoid(b)  reach at T=80
         0.0      0.5000       1.20e-18
         1.0      0.7311       3.02e-09
         2.0      0.8808       2.42e-03
         4.0      0.9820       2.23e-02
         6.0      0.9975       5.94e-02


**Observation:** Prediction was wrong. Reach kept increasing with the bias: 1.20e-18 (b=0), 3.02e-09 (b=1), 2.42e-03 (b=2), 2.23e-02 (b=4) and 5.94e-02 (b=6). A larger bias keeps the forget gate closer to 1 and the gradient path more open. The downside I predicted, a gate that never forgets, is a cost on the task and cannot show up in a gradient-reach measurement.

## **Exercise 4 — Add a GRU to the recall task (Experiment 3.3)**

**Prediction first (write it down before running):** how will a default GRU compare with the default LSTM, which scored 51 %?

**Prediction:** In `nn.GRU`, $z$ weights the **old** state, and default gates start near $\sigma(0)=0.5$, so the old state is halved at every step, just like the default LSTM's forget gate. I expect the GRU to **also fail at T=60** (near chance, within roughly 50–70 %) and to be rescued only by an initialisation that pushes $z$ towards 1.

In [19]:
class RecallGRU(nn.Module):
    def __init__(self, z_bias=0.0, d_h=24):
        super().__init__()
        self.rnn = nn.GRU(2, d_h, batch_first=True)

        # PyTorch GRU bias blocks are ordered r, z, n; z is the SECOND block
        with torch.no_grad():
            self.rnn.bias_ih_l0[d_h:2 * d_h].fill_(z_bias)
            self.rnn.bias_hh_l0[d_h:2 * d_h].fill_(0.0)

        self.fc = nn.Linear(d_h, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1]).squeeze(-1)


def train_recall(model, T=60, steps=400):
    gen = torch.Generator().manual_seed(1)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)

    for _ in range(steps):
        x, y = make_batch(64, T, gen)

        opt.zero_grad()
        F.binary_cross_entropy_with_logits(model(x), y).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

    x, y = make_batch(512, T, torch.Generator().manual_seed(99))

    with torch.no_grad():
        return ((model(x) > 0).float() == y).float().mean().item()


for label, build in [("LSTM, default", lambda: Recall("lstm", 0.0)),
                     ("GRU, default", lambda: RecallGRU(0.0)),
                     ("GRU, z bias 2", lambda: RecallGRU(2.0))]:
    torch.manual_seed(0)
    print(f"{label:>16}: test accuracy {100 * train_recall(build()):5.1f}%")

   LSTM, default: test accuracy  51.0%
    GRU, default: test accuracy  51.4%
   GRU, z bias 2: test accuracy 100.0%


**Observation:** Prediction held. The default GRU scored 51.4%, the same near-chance result as the default LSTM (51.0%), because the update gate starts near 0.5 and the old state is halved at every step. Setting the z bias to 2 raised it to 100%.

## **Exercise 5 — Label is the bit at position t − 1 (Experiment 5.2)**

**Prediction first:** what do the unidirectional LSTM and the BiLSTM score now, and what happens to the BiLSTM on a stream?

**Prediction:** The label now depends only on the **past**, so a unidirectional LSTM can solve it: about **100 %**. The BiLSTM also reaches about 100 % with the whole sequence visible. On a stream the future is zero-padded, which the BiLSTM never saw in training, so its backward half feeds in unfamiliar input; it may lose some accuracy, but far less than in 5.2, because the answer no longer depends on the future.

In [20]:
def make_prev_batch(n, T, gen):
    # Label at position t is the token at position t - 1
    tok = torch.randint(0, 2, (n, T), generator=gen).float()

    y = torch.zeros(n, T)
    y[:, 1:] = tok[:, :-1]            # shift RIGHT by one

    return tok.unsqueeze(-1), y


def train_prev(model, steps=300):
    gen = torch.Generator().manual_seed(1)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)

    for _ in range(steps):
        x, y = make_prev_batch(64, T, gen)

        opt.zero_grad()
        # position 0 has no previous token, so it is left out of the loss
        F.binary_cross_entropy_with_logits(model(x)[:, 1:], y[:, 1:]).backward()
        opt.step()

    return model


x, y = make_prev_batch(512, T, torch.Generator().manual_seed(99))

for label, bi in (("unidirectional LSTM", False), ("BiLSTM", True)):
    torch.manual_seed(0)
    model = train_prev(Tagger(bi))

    with torch.no_grad():
        acc = ((model(x)[:, 1:] > 0).float() == y[:, 1:]).float().mean().item()

    print(f"{label:>20}: test accuracy {100 * acc:5.1f}%")

# ---- the trained BiLSTM on a stream (future zero-padded) ----
hits = total = 0

with torch.no_grad():
    for t in range(1, T):
        seen = x.clone()
        seen[:, t + 1:, :] = 0.0

        hits += ((model(seen)[:, t] > 0).float() == y[:, t]).sum().item()
        total += y.shape[0]

print(f"{'BiLSTM on a stream':>20}: test accuracy {100 * hits / total:5.1f}%")

 unidirectional LSTM: test accuracy 100.0%
              BiLSTM: test accuracy 100.0%
  BiLSTM on a stream: test accuracy 100.0%


**Observation:** Prediction held for the first two results. The unidirectional LSTM and the BiLSTM both scored 100%. The BiLSTM also scored 100% on a stream, which is better than the small drop I expected. The label depends only on past tokens, so zero-padding the future does not hurt, unlike Experiment 5.2 where the stream score fell from 100% to 49%.

# **Viva Questions**

**1. Why must the gradient of a shared weight be summed over time steps?**
- The same $W_{hh}$ is used at **every** step, so a change in it affects the loss through every step. By the chain rule the total derivative is the **sum** of the contributions from each use: $\partial L/\partial W_{hh} = \sum_t \delta_t h_{t-1}^T$.

**2. Why does clipping help with exploding gradients but not with vanishing ones?**
- Clipping rescales a gradient whose norm is **too large** and keeps its direction, so it caps explosions.
- A vanished gradient (say 1e-12) carries almost no information; scaling it up would amplify numerical noise. Vanishing needs an **architectural** fix: gating (LSTM, GRU) or a better initialisation.

**3. What does the additive cell update give an LSTM that a plain RNN lacks, and why did the default LSTM still fail at 60 steps?**
- $c_t = f_t\odot c_{t-1} + i_t\odot g_t$ is an **additive** path, so the gradient flows back through the product of forget gates instead of repeated $\tanh$ and $W_{hh}$ multiplications.
- With default initialisation $f\approx0.5$, so over 60 steps the gradient is scaled by about $0.5^{60}$ and almost none reaches step 0. Setting the forget bias to 1 or 2 opens the path ($f\approx0.73$ or $0.88$): the architecture supplies the path, the **gate values** decide whether it is open.

**4. Name the two differences between the textbook GRU and `nn.GRU`.**
- **Update gate meaning:** in PyTorch $z$ weights the *old* state ($h_t=(1-z)n+zh_{t-1}$), the textbook weights the candidate.
- **Reset gate position:** PyTorch applies $r$ *after* the recurrent matrix, $r\odot(Uh+b)$; the textbook applies it *before*, $U(r\odot h)$.

**5. Why is `out[:, -1]` not the sequence representation of a BiLSTM, and what should be used instead?**
- The forward half at the last position has read the whole sequence, but the **backward half at the last position has read only one token**, so the vector is badly wrong (58 % off in Experiment 5.1) and nothing raises an error.
- Use `torch.cat([h_n[0], h_n[1]], dim=1)` (forward final state + backward final state) for sequence-level tasks; the full `out` tensor is right for per-position labelling.

**6. A colleague reports 100 % accuracy from a BiLSTM forecasting the next value of a live sensor stream. What is your first question?**
- **"Did the model see the future values when it made each prediction?"** A BiLSTM needs the whole sequence, so on a live stream it cannot; a score measured on complete series reflects the model *reading* the answer. Experiment 5.2 shows the same weights falling from 100 % to 49 % once the future is withheld.